<a href="https://colab.research.google.com/github/shoukk8-afk/Titanic-NN/blob/main/Titanic_(NN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
df = pd.read_csv("train.csv")

In [ ]:
del df["Ticket"]

In [ ]:
df['Fare'] = df['Fare'].apply(np.log1p)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df["member_num"] = df["SibSp"] + df["Parch"] + 1

In [ ]:
dummy_sex = pd.get_dummies(df["Sex"], dtype=int, drop_first=True)

In [ ]:
def extract_title(df):
    # 1. 正規表現で「空白 + 文字列 + ピリオド」のパターンを抽出
    # 例: "Braund, Mr. Owen Harris" -> "Mr"
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

    # 2. 統合ルールの定義
    # Mr, Miss, Mrs 以外（+ 子供の Master）を 'Rare' にまとめる
    title_mapping = {
        "Mr": "Mr",
        "Miss": "Miss",
        "Mrs": "Mrs",
        "Master": "Master",  # 子供は生存率が高いので残すのが定石
        "Mlle": "Miss",      # フランス語の未婚女性
        "Ms": "Miss",
        "Mme": "Mrs",       # フランス語の既婚女性
    }

    # 3. マッピングを適用し、それ以外を 'Rare' にする
    df['Title'] = df['Title'].map(title_mapping).fillna('Rare')

    return df

In [ ]:
df = extract_title(df)
title = pd.get_dummies(df["Title"], dtype=int, drop_first=True)
title.head()

,Miss,Mr,Mrs,Rare
0,0,1,0,0
1,0,0,1,0
2,1,0,0,0
3,0,0,1,0
4,0,1,0,0


In [ ]:
embark = pd.get_dummies(df["Embarked"], drop_first=True, dtype=int)

In [ ]:
df = df.join([dummy_sex, title, embark])

In [ ]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Fare', 'Cabin', 'Embarked', 'member_num', 'Title', 'male',
       'Miss', 'Mr', 'Mrs', 'Rare', 'Q', 'S'],
      dtype='object')

In [ ]:
df['Age'] = df['Age'].fillna(df.groupby('Title')['Age'].transform('median'))

In [ ]:
df = df.drop(columns=["Name", "Sex", "Cabin", "Embarked", "Title"])

In [ ]:
df = df.set_index("PassengerId")

In [ ]:
df_train_features = df.iloc[:, 1:]
df_train_label = df.iloc[:, 0]

In [ ]:
test = pd.read_csv("test.csv")

In [ ]:
test['Fare'] = test['Fare'].apply(np.log1p)

In [ ]:
test["member_num"] = test["SibSp"] + test["Parch"] + 1

In [ ]:
test = extract_title(test)
test_title = pd.get_dummies(test["Title"], dtype=int, drop_first=True)
test_title.head()

,Miss,Mr,Mrs,Rare
0,0,1,0,0
1,0,0,1,0
2,0,1,0,0
3,0,1,0,0
4,0,0,1,0


In [ ]:
test_embark = pd.get_dummies(test["Embarked"], dtype=int, drop_first=True)

In [ ]:
test_dummy = pd.get_dummies(test["Sex"], dtype=int, drop_first=True)

In [ ]:
test = test.join([test_dummy, test_title, test_embark])
test.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked', 'member_num', 'Title', 'male',
       'Miss', 'Mr', 'Mrs', 'Rare', 'Q', 'S'],
      dtype='object')

In [ ]:
test['Age'] = test['Age'].fillna(test.groupby('Title')['Age'].transform('median'))

In [ ]:
test = test.drop(columns=["Name", "Sex", "Ticket", "Cabin", "Embarked", "Title"])

In [ ]:
test = test.set_index("PassengerId")

In [ ]:
df.columns

Index(['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'member_num',
       'male', 'Miss', 'Mr', 'Mrs', 'Rare', 'Q', 'S'],
      dtype='object')

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_train_features_scaled = scaler.fit_transform(df_train_features)

In [ ]:
seq_model = nn.Sequential(
    nn.Linear(13, 52),
    nn.ReLU(),
    nn.Linear(52, 13),
    nn.ReLU(),
    nn.Linear(13, 1),
    nn.Sigmoid()
)

loss_fn = nn.BCELoss()
optimizer = optim.SGD(seq_model.parameters(), lr=1e-2)

In [ ]:
X_train = torch.tensor(df_train_features_scaled, dtype=torch.float32)
y_train = torch.tensor(df_train_label.values, dtype=torch.float32).view(-1, 1)

X_train.shape

torch.Size([891, 13])

In [ ]:
def training_loop(n_epochs, optimizer, model, loss_fn, X_train, y_train):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = loss_fn(y_pred, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch}: Training loss {loss.item():.4f}")

In [ ]:
summary = scaler.fit(df_train_features)
test_scaled = scaler.transform(test)
X_test = torch.tensor(test_scaled, dtype=torch.float32)

In [ ]:
with torch.no_grad():
    y_test = seq_model(X_test)
y_preds = (y_test >= 0.5).int()

In [ ]:
y_preds.shape

torch.Size([418, 1])

In [ ]:
X_test.shape

torch.Size([418, 13])

In [ ]:
test["Survived"] = y_preds.numpy()

In [ ]:
test = test.reset_index()
submission = test[["PassengerId", "Survived"]]
submission.to_csv("submission.csv", index=False)